In [1]:
%pip install numpy matplotlib ipywidgets pandas openpyxl


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets

from IPython.display import display, clear_output

In [3]:
AFM_FOLDER = Path("afm_data")

EXCEL_FOLDER = Path("afm_values")

DEFAULT_X_MIN = -150

DEFAULT_X_MAX = 200

def load_afm_txt(file_path):

    segments = {}

    current_segment = None

    with open(
        file_path,
        "r",
        encoding="utf-8",
        errors="ignore"
    ) as file:

        for line in file:

            line = line.strip()


            # -----------------------------------------------
            # Identify approach or retract section
            # -----------------------------------------------

            if line.startswith("# segment:"):

                current_segment = (
                    line
                    .split(":", 1)[1]
                    .strip()
                    .lower()
                )

                segments.setdefault(
                    current_segment,
                    []
                )

                continue


            # -----------------------------------------------
            # Ignore empty lines
            # -----------------------------------------------

            if not line:
                continue


            # -----------------------------------------------
            # Ignore header/comment lines
            # -----------------------------------------------

            if line.startswith("#"):
                continue


            # -----------------------------------------------
            # Ignore anything before the first segment
            # -----------------------------------------------

            if current_segment is None:
                continue


            # -----------------------------------------------
            # Convert data to numbers
            # -----------------------------------------------

            try:

                row = [
                    float(value)
                    for value in line.split()
                ]

            except ValueError:

                continue


            # -----------------------------------------------
            # Store rows containing at least X and Y
            # -----------------------------------------------

            if len(row) >= 2:

                segments[
                    current_segment
                ].append(row)


    # --------------------------------------------------------
    # Convert to NumPy arrays
    # --------------------------------------------------------

    approach = np.asarray(
        segments.get("extend", []),
        dtype=float
    )

    retract = np.asarray(
        segments.get("retract", []),
        dtype=float
    )

    return approach, retract


# ============================================================
# EXTRACT CURVE NUMBER FROM AFM FILE NAME
# ============================================================
#
# Example filename:
#
# qi-data-2024.10.02-16.49.20.279-cropped_032.txt
#
# returns:
#
# 32
#
# ============================================================

def extract_curve_number(file_path):

    filename = file_path.name

    # Search for digits immediately before .txt
    match = re.search(
        r"_(\d+)\.txt$",
        filename,
        re.IGNORECASE
    )

    if match:

        return int(
            match.group(1)
        )

    return None


# ============================================================
# DETERMINE EXCEL FILE AND SHEET FROM AFM FOLDER
# ============================================================
#
# Example AFM folder:
#
# +ve_cell1part2
#
# gives:
#
# condition = +ve
# sheet     = cell1part2
#
# therefore:
#
# Excel file = +ve.xlsx
# Excel sheet = cell1part2
#
# ============================================================

def get_excel_location(afm_file):

    folder_name = (
        afm_file.parent.name
    )


    # Split only at the first underscore
    #
    # +ve_cell1part2
    #
    # becomes:
    #
    # +ve
    # cell1part2

    if "_" not in folder_name:

        return None, None


    condition, sheet_name = (
        folder_name.split(
            "_",
            1
        )
    )


    # Build Excel filename
    #
    # +ve -> +ve.xlsx

    excel_file = (
        EXCEL_FOLDER
        /
        f"{condition}.xlsx"
    )


    return excel_file, sheet_name


# ============================================================
# CACHE EXCEL SHEETS
# ============================================================
#
# Reading Excel files repeatedly is slower.
#
# This dictionary stores sheets after they have been read once.
#
# ============================================================

excel_cache = {}


# ============================================================
# READ E, kB AND RMS FOR SELECTED CURVE
# ============================================================

def get_curve_values(afm_file):


    # --------------------------------------------------------
    # Find curve number from filename
    # --------------------------------------------------------

    curve_number = (
        extract_curve_number(
            afm_file
        )
    )


    if curve_number is None:

        return {
            "curve": None,
            "E": None,
            "kB": None,
            "RMS": None,
            "status": "Curve number not found"
        }


    # --------------------------------------------------------
    # Determine corresponding Excel file and sheet
    # --------------------------------------------------------

    excel_file, sheet_name = (
        get_excel_location(
            afm_file
        )
    )


    if excel_file is None:

        return {
            "curve": curve_number,
            "E": None,
            "kB": None,
            "RMS": None,
            "status": "Excel mapping not found"
        }


    # --------------------------------------------------------
    # Check Excel file exists
    # --------------------------------------------------------

    if not excel_file.exists():

        return {
            "curve": curve_number,
            "E": None,
            "kB": None,
            "RMS": None,
            "status": f"Missing {excel_file.name}"
        }


    # --------------------------------------------------------
    # Create cache key
    # --------------------------------------------------------

    cache_key = (
        str(excel_file),
        sheet_name
    )


    # --------------------------------------------------------
    # Read Excel sheet only if it has not already been loaded
    # --------------------------------------------------------

    if cache_key not in excel_cache:

        try:

            df = pd.read_excel(
                excel_file,
                sheet_name=sheet_name
            )

            excel_cache[
                cache_key
            ] = df

        except Exception as error:

            return {
                "curve": curve_number,
                "E": None,
                "kB": None,
                "RMS": None,
                "status": f"Excel error: {error}"
            }


    # --------------------------------------------------------
    # Get cached Excel data
    # --------------------------------------------------------

    df = excel_cache[
        cache_key
    ].copy()


    # --------------------------------------------------------
    # Make sure Curve column exists
    # --------------------------------------------------------

    if "Curve" not in df.columns:

        return {
            "curve": curve_number,
            "E": None,
            "kB": None,
            "RMS": None,
            "status": "Curve column missing"
        }


    # --------------------------------------------------------
    # Convert Curve column to numbers
    #
    # Invalid values become NaN.
    # --------------------------------------------------------

    df["Curve"] = pd.to_numeric(
        df["Curve"],
        errors="coerce"
    )


    # --------------------------------------------------------
    # Find selected curve
    # --------------------------------------------------------

    row = df[
        df["Curve"] == curve_number
    ]


    # --------------------------------------------------------
    # Curve does not exist in Excel
    # --------------------------------------------------------

    if row.empty:

        return {
            "curve": curve_number,
            "E": None,
            "kB": None,
            "RMS": None,
            "status": "Curve not found in Excel"
        }


    # --------------------------------------------------------
    # Use first matching row
    # --------------------------------------------------------

    row = row.iloc[0]


    # --------------------------------------------------------
    # Helper function
    #
    # Converts valid numbers to float.
    #
    # #VALUE!, NaN, blanks etc become None.
    # --------------------------------------------------------

    def safe_number(value):

        try:

            number = float(value)

            if np.isfinite(number):

                return number

        except:

            pass

        return None


    # --------------------------------------------------------
    # Read columns from your Excel format
    # --------------------------------------------------------

    E = safe_number(
        row.get(
            "E (MPa)"
        )
    )

    kB = safe_number(
        row.get(
            "kB (nN/µm)"
        )
    )

    RMS = safe_number(
        row.get(
            "RMS (pN)"
        )
    )


    return {

        "curve": curve_number,

        "E": E,

        "kB": kB,

        "RMS": RMS,

        "status": "OK",

        "excel_file": excel_file.name,

        "sheet": sheet_name
    }


# ============================================================
# FORMAT NUMBERS FOR DISPLAY
# ============================================================

def format_value(
    value,
    decimals=3
):

    if value is None:

        return "N/A"

    return f"{value:.{decimals}f}"


# ============================================================
# PLOT ONE AFM CURVE
# ============================================================

def plot_afm(
    file_path,
    x_min=-150,
    x_max=200
):


    # --------------------------------------------------------
    # Read AFM curve
    # --------------------------------------------------------

    approach, retract = (
        load_afm_txt(
            file_path
        )
    )


    # --------------------------------------------------------
    # Read matching Excel values
    # --------------------------------------------------------

    values = (
        get_curve_values(
            file_path
        )
    )


    # --------------------------------------------------------
    # Create figure
    #
    # Extra width is used for the E/kB/RMS information panel.
    # --------------------------------------------------------

    fig, ax = plt.subplots(
        figsize=(10, 5.5)
    )


    # Leave space on right side
    # for parameter information

    fig.subplots_adjust(
        right=0.72
    )


    # ========================================================
    # APPROACH CURVE
    # ========================================================

    if approach.size > 0:


        # Convert metres to nm

        x_approach = (
            approach[:, 0]
            *
            1e9
        )


        # Convert Newtons to nN

        y_approach = (
            approach[:, 1]
            *
            1e9
        )


        # Restrict displayed range

        mask = (

            (x_approach >= x_min)

            &

            (x_approach <= x_max)

        )


        # Plot approach in black

        ax.plot(

            x_approach[mask],

            y_approach[mask],

            color="black",

            linewidth=1.2,

            label="Approach"
        )


    # ========================================================
    # RETRACT CURVE
    # ========================================================

    if retract.size > 0:


        # Convert metres to nm

        x_retract = (
            retract[:, 0]
            *
            1e9
        )


        # Convert Newtons to nN

        y_retract = (
            retract[:, 1]
            *
            1e9
        )


        # Restrict displayed range

        mask = (

            (x_retract >= x_min)

            &

            (x_retract <= x_max)

        )


        # Plot retract in red

        ax.plot(

            x_retract[mask],

            y_retract[mask],

            color="red",

            linewidth=1.2,

            label="Retract"
        )


    # ========================================================
    # GRAPH FORMATTING
    # ========================================================

    ax.set_xlim(
        x_min,
        x_max
    )


    ax.set_xlabel(
        "Indentation (nm)",
        fontsize=12
    )


    ax.set_ylabel(
        "Force (nN)",
        fontsize=12
    )


    ax.set_title(
        file_path.name,
        fontsize=10
    )


    ax.legend()


    # ========================================================
    # CREATE E / kB / RMS INFORMATION PANEL
    # ========================================================

    E_text = format_value(
        values.get("E"),
        3
    )

    kB_text = format_value(
        values.get("kB"),
        3
    )

    RMS_text = format_value(
        values.get("RMS"),
        2
    )


    curve_number = (
        values.get("curve")
    )


    if curve_number is None:

        curve_text = "N/A"

    else:

        curve_text = (
            f"_{curve_number:03d}"
        )


    parameter_text = (

        f"Curve: {curve_text}\n\n"

        f"kB\n"
        f"{kB_text} nN/µm\n\n"

        f"E\n"
        f"{E_text} MPa\n\n"

        f"RMS\n"
        f"{RMS_text} pN"

    )


    # --------------------------------------------------------
    # Put information on right side of graph
    # --------------------------------------------------------

    fig.text(

        0.76,

        0.72,

        parameter_text,

        fontsize=12,

        verticalalignment="top",

        bbox=dict(

            boxstyle="round",

            facecolor="white",

            edgecolor="gray",

            alpha=0.9
        )
    )


    plt.show()


    return (
        approach,
        retract,
        values
    )


# ============================================================
# FIND ALL AFM SUBFOLDERS
# ============================================================

folders = sorted({

    file.parent

    for file
    in AFM_FOLDER.rglob("*.txt")

})


# ============================================================
# CHECK AFM DATA EXISTS
# ============================================================

if len(folders) == 0:

    raise FileNotFoundError(

        f"No AFM .txt files found inside:\n"
        f"{AFM_FOLDER.resolve()}"

    )


# ============================================================
# FOLDER DROPDOWN
# ============================================================

folder_dropdown = widgets.Dropdown(

    options=[

        (

            str(
                folder.relative_to(
                    AFM_FOLDER
                )
            ),

            folder

        )

        for folder in folders

    ],

    description="Folder:",

    layout=widgets.Layout(
        width="700px"
    )
)


# ============================================================
# CURVE DROPDOWN
# ============================================================

file_dropdown = widgets.Dropdown(

    description="Curve:",

    layout=widgets.Layout(
        width="700px"
    )
)


# ============================================================
# BUTTONS
# ============================================================

previous_button = widgets.Button(

    description="Previous"
)


next_button = widgets.Button(

    description="Next"
)


# ============================================================
# X AXIS CONTROLS
# ============================================================

x_min_box = widgets.FloatText(

    value=DEFAULT_X_MIN,

    description="X min (nm):"
)


x_max_box = widgets.FloatText(

    value=DEFAULT_X_MAX,

    description="X max (nm):"
)


# ============================================================
# OUTPUT AREA
# ============================================================

plot_output = widgets.Output()

info_output = widgets.Output()


# ============================================================
# GET CURVES IN SELECTED FOLDER
# ============================================================
#
# reverse=True gives last curve first.
#
# ============================================================

def get_files_from_folder(folder):

    files = sorted(

        folder.glob("*.txt"),

        reverse=True
    )

    return files


# ============================================================
# UPDATE CURVE DROPDOWN
# ============================================================

def update_file_dropdown(change=None):

    folder = (
        folder_dropdown.value
    )

    files = (
        get_files_from_folder(
            folder
        )
    )


    file_dropdown.options = [

        (
            file.name,
            file
        )

        for file in files

    ]


    if len(files) > 0:

        file_dropdown.value = (
            files[0]
        )


# ============================================================
# REDRAW SELECTED CURVE
# ============================================================

def redraw_plot(change=None):


    file_path = (
        file_dropdown.value
    )


    if file_path is None:

        return


    # --------------------------------------------------------
    # Plot graph
    # --------------------------------------------------------

    with plot_output:

        clear_output(
            wait=True
        )


        approach, retract, values = (
            plot_afm(

                file_path,

                x_min=x_min_box.value,

                x_max=x_max_box.value
            )
        )


    # --------------------------------------------------------
    # Display matching information below controls
    # --------------------------------------------------------

    with info_output:

        clear_output(
            wait=True
        )


        folder = (
            folder_dropdown.value
        )


        files = (
            get_files_from_folder(
                folder
            )
        )


        current_index = (
            files.index(
                file_path
            )
        )


        print(
            "Folder:",
            folder.relative_to(
                AFM_FOLDER
            )
        )


        print(
            "File:",
            file_path.name
        )


        print(
            f"Curve {current_index + 1} "
            f"of {len(files)}"
        )


        print()


        # ----------------------------------------------------
        # Corresponding Excel information
        # ----------------------------------------------------

        if values.get(
            "status"
        ) == "OK":

            print(
                "Excel:",
                values.get(
                    "excel_file"
                )
            )

            print(
                "Sheet:",
                values.get(
                    "sheet"
                )
            )

            print(
                "Curve number:",
                values.get(
                    "curve"
                )
            )

            print(
                "E:",
                format_value(
                    values.get("E"),
                    3
                ),
                "MPa"
            )

            print(
                "kB:",
                format_value(
                    values.get("kB"),
                    3
                ),
                "nN/µm"
            )

            print(
                "RMS:",
                format_value(
                    values.get("RMS"),
                    2
                ),
                "pN"
            )

        else:

            print(
                "Excel status:",
                values.get(
                    "status"
                )
            )


# ============================================================
# PREVIOUS CURVE
# ============================================================

def previous_curve(button):


    folder = (
        folder_dropdown.value
    )


    files = (
        get_files_from_folder(
            folder
        )
    )


    current_file = (
        file_dropdown.value
    )


    if current_file is None:

        return


    current_index = (
        files.index(
            current_file
        )
    )


    new_index = max(

        0,

        current_index - 1
    )


    file_dropdown.value = (
        files[new_index]
    )


# ============================================================
# NEXT CURVE
# ============================================================

def next_curve(button):


    folder = (
        folder_dropdown.value
    )


    files = (
        get_files_from_folder(
            folder
        )
    )


    current_file = (
        file_dropdown.value
    )


    if current_file is None:

        return


    current_index = (
        files.index(
            current_file
        )
    )


    new_index = min(

        len(files) - 1,

        current_index + 1
    )


    file_dropdown.value = (
        files[new_index]
    )


# ============================================================
# CONNECT BUTTONS
# ============================================================

previous_button.on_click(
    previous_curve
)


next_button.on_click(
    next_curve
)


# ============================================================
# FOLDER CHANGE
# ============================================================

def folder_changed(change):

    update_file_dropdown()


folder_dropdown.observe(

    folder_changed,

    names="value"
)


# ============================================================
# CURVE CHANGE
# ============================================================

file_dropdown.observe(

    redraw_plot,

    names="value"
)


# ============================================================
# X AXIS CHANGE
# ============================================================

x_min_box.observe(

    redraw_plot,

    names="value"
)


x_max_box.observe(

    redraw_plot,

    names="value"
)


# ============================================================
# INITIALISE CURVES
# ============================================================

update_file_dropdown()


# ============================================================
# CREATE INTERFACE
# ============================================================

navigation = widgets.HBox([

    previous_button,

    next_button
])


axis_controls = widgets.HBox([

    x_min_box,

    x_max_box
])


controls = widgets.VBox([

    folder_dropdown,

    file_dropdown,

    navigation,

    axis_controls

])


# ============================================================
# DISPLAY VIEWER
# ============================================================

display(

    controls,

    info_output,

    plot_output
)


# ============================================================
# DISPLAY FIRST CURVE
# ============================================================

redraw_plot()

Output()

Output()